In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import os
import time
from building_models.utils.utils_functions import UtilsFunctions

In [6]:
path_data = "../../processed_dataset/antioxidant_classification/"
path_export = "../../processed_dataset/"

In [7]:
list_sources = os.listdir(path_data)
list_sources = [value for value in list_sources if value != "processed_dataset"]
list_sources

['Lam et al.', 'Zhai et al.', 'PredAoDP', 'ANOX', 'Zhang et al.']

In [8]:
list_dfs = []

for source in list_sources:
    df = pd.read_csv(f"{path_data}/{source}/processed_data.csv")
    df["source"] = source
    list_dfs.append(df[["sequence", "label", "source"]])

df_all = pd.concat(list_dfs, ignore_index=True)

df_pivot = (
    df_all.pivot_table(
        index="sequence",
        columns="source",
        values="label",
        aggfunc="first"
    )
    .reset_index()
)

df_pivot.columns.name = None
df_pivot = df_pivot.fillna(999)

for column in df_pivot.columns:
    if column != "sequence":
        df_pivot[column] = df_pivot[column].astype(int)
print(df_pivot.shape)
df_pivot.head()

(2381, 6)


,sequence,ANOX,Lam et al.,PredAoDP,Zhai et al.,Zhang et al.
0,AAAAAAMTMMDMNFKYCHKIMKKHSKSFSYAFDLLPEDQRKAVWAI...,0,0,0,0,999
1,AAASFGQTKIPRGNGPYSVGCTDLMFDHTNKGTFLRLYYPSQDNDR...,0,0,0,0,999
2,AACYSSDCRVKCVAMGFSSGKCINSKCKCYK,0,999,0,0,999
3,AAGTAKGHNPTEFPTIYDASSAPTAANTTVGIITIGGVSQTLQDLQ...,0,0,0,0,999
4,AAKELTLAQTESLREVCETNMACDEMADAQGIVAAYQAFYGPIPF,0,0,0,0,999


In [9]:
source_cols = [col for col in df_pivot.columns if col != "sequence"]

df_pivot["count_0"] = (df_pivot[source_cols] == 0).sum(axis=1)
df_pivot["count_1"] = (df_pivot[source_cols] == 1).sum(axis=1)

df_pivot["n_valid_sources"] = df_pivot[source_cols].isin([0, 1]).sum(axis=1)

df_pivot["count_0_norm"] = (
    df_pivot["count_0"] / df_pivot["n_valid_sources"].replace(0, pd.NA)
)

df_pivot["count_1_norm"] = (
    df_pivot["count_1"] / df_pivot["n_valid_sources"].replace(0, pd.NA)
)

df_pivot.head()

,sequence,ANOX,Lam et al.,PredAoDP,Zhai et al.,Zhang et al.,count_0,count_1,n_valid_sources,count_0_norm,count_1_norm
0,AAAAAAMTMMDMNFKYCHKIMKKHSKSFSYAFDLLPEDQRKAVWAI...,0,0,0,0,999,4,0,4,1.0,0.0
1,AAASFGQTKIPRGNGPYSVGCTDLMFDHTNKGTFLRLYYPSQDNDR...,0,0,0,0,999,4,0,4,1.0,0.0
2,AACYSSDCRVKCVAMGFSSGKCINSKCKCYK,0,999,0,0,999,3,0,3,1.0,0.0
3,AAGTAKGHNPTEFPTIYDASSAPTAANTTVGIITIGGVSQTLQDLQ...,0,0,0,0,999,4,0,4,1.0,0.0
4,AAKELTLAQTESLREVCETNMACDEMADAQGIVAAYQAFYGPIPF,0,0,0,0,999,4,0,4,1.0,0.0


In [10]:
df_pivot_positive = df_pivot[df_pivot["count_1_norm"] == 1]
df_pivot_negative = df_pivot[df_pivot["count_0_norm"] == 1]

df_pivot_positive.shape, df_pivot_negative.shape

((323, 11), (2015, 11))

In [11]:
df_pivot_positive["label"] = 1
df_pivot_negative["label"] = 0

In [12]:
df_processed = pd.concat([df_pivot_positive, df_pivot_negative], axis=0, ignore_index=True)
df_processed["label"].value_counts()

label
0    2015
1     323
Name: count, dtype: int64

In [13]:
sequences_with_different_annotations = df_pivot[~df_pivot["sequence"].isin(df_processed["sequence"])]
print(sequences_with_different_annotations.shape)
sequences_with_different_annotations.head()

(43, 11)


,sequence,ANOX,Lam et al.,PredAoDP,Zhai et al.,Zhang et al.,count_0,count_1,n_valid_sources,count_0_norm,count_1_norm
702,MAGQKIRIRLKAYDHEAIDASARKIVETVVRTGASVVGPVPLPTEK...,999,999,1,999,0,1,1,2,0.5,0.5
716,MAHKKAGGSTRNGRDSESKRLGVKRFGGESVLAGNIIVRQRGTKFH...,999,999,1,999,0,1,1,2,0.5,0.5
719,MAIDENKQKALAAALGQIEKQFGKGSIMRLGEDRSMDVETISTGSL...,999,999,1,999,0,1,1,2,0.5,0.5
729,MAKISKRRQAFAAKVDRQKLYAIEDALSLVKECASAKFDESIDVAV...,999,999,1,999,0,1,1,2,0.5,0.5
795,MARDIAAPPVPTNHQELISWVNEIAELTQPDAVVWCDGSEAEYERL...,999,999,1,999,0,1,1,2,0.5,0.5


- Exporting data

In [15]:
os.makedirs(f"{path_export}/merged_data", exist_ok=True)

In [16]:
df_processed["length"] = df_processed["sequence"].str.len()

In [17]:
df_processed.to_csv(f"{path_export}/merged_data/processed_dataset.csv", index=False)
sequences_with_different_annotations.to_csv(f"{path_export}/merged_data/sequences_with_erros.csv", index=False)

- Checking unlabeled data from Lam et al

In [18]:
df_unknown = pd.read_csv(f"{path_data}/Lam et al./unlabeled_data.csv")
df_unknown["is_in_procesed"] = df_unknown["sequence"].isin(df_processed["sequence"].values)
df_unknown["is_in_procesed"].value_counts()

is_in_procesed
False    110
True      10
Name: count, dtype: int64

In [19]:
df_unknown.to_csv(f"{path_export}/merged_data/sequences_without_label.csv", index=False)

- Create metadata

In [20]:
dict_metadata = {
    "task" : "antioxidant_classification",
    "mode" : "binary",
    "number_of_examples": df_pivot.shape[0],
    "number_of_inconsistences" : sequences_with_different_annotations.shape[0],
    "number_of_sequences_without_label" : df_unknown.shape[0],
    "number_unique_annotated" : df_processed.shape[0],
    "positive_examples" : df_pivot_positive.shape[0],
    "negative_examples" : df_pivot_negative.shape[0],
    "statistic_dataset": {
        "min_length": int(df_processed["length"].min()),
        "max_length" : int(df_processed["length"].max())
    },
    "date_process" : time.strftime("%Y-%m-%d %H:%M:%S")
}

dict_metadata

{'task': 'antioxidant_classification',
 'mode': 'binary',
 'number_of_examples': 2381,
 'number_of_inconsistences': 43,
 'number_of_sequences_without_label': 120,
 'number_unique_annotated': 2338,
 'positive_examples': 323,
 'negative_examples': 2015,
 'statistic_dataset': {'min_length': 11, 'max_length': 2306},
 'date_process': '2026-04-08 09:42:15'}

In [21]:
UtilsFunctions.export_json(f"{path_export}/merged_data/metadata.json", dict_metadata)